# Data Preparation and cleaning:-

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import scipy
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

sys.version

# E-commerce Data Preparation and Feature Engineering

## 1. Environment Setup and Library Verification
## 2. Data Loading
## 3. Data Understanding
## 4. Data Cleaning
## 5. Data Transformation
## 6. Feature Engineering
## 7. Final Cleaned Dataset

#1. Loading he dataset (Keep the file in the same directory as this notebook or provide the correct path)
df = pd.read_excel(r"Online Retail.xlsx")

print('\n🔴 Shape of dataset 🔴', df.shape)
display(df.head())

#3️ Understand the Data
print("\n🔴 Dataset Info 🔴")
df.info()

# 4️ Statistical Summary
print("\n🔴 Statistical Summary 🔴")
display(df.describe())

# Missing Values
print("\n🔴 Missing values per column 🔴")
display(df.isnull().sum())

# Duplicate rows
print("\n🔴 Duplicate rows 🔴", df.duplicated().sum())

#4️⃣ Data Cleaning
#4.1 Remove Duplicate Records
df.drop_duplicates(inplace=True)

#4.2 Remove Missing Customer IDs
df = df[df['CustomerID'].notnull()]

#4.3 Remove Invalid Quantity and Price  
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

#4.4 Correct Data Types
df['CustomerID'] = df['CustomerID'].astype(int)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

#5️⃣ Post-Cleaning Validation
print("🔴 Shape after cleaning 🔴", df.shape)

# Missing values per column after cleaning
print("\n🔴 Missing values per column after cleaning 🔴")
display(df.isnull().sum())

# Duplicate rows after cleaning
print("\n🔴 Duplicate rows after cleaning 🔴", df.duplicated().sum())
display(df.head())

#5️⃣ Feature Engineering
#5.1 Create New Features
#Total Purchase Value
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

#Invoice Day & Month (Time-based Features)
df['InvoiceDay'] = df['InvoiceDate'].dt.day
df['InvoiceMonth'] = df['InvoiceDate'].dt.month

#Customer-Level Aggregation (RFM-style Features)
customer_df = df.groupby('CustomerID').agg({
    'InvoiceNo': 'nunique',      # Frequency
    'Quantity': 'sum',           # Total quantity purchased
    'TotalPrice': 'sum',         # Monetary value
    'InvoiceDate': 'max'         # Recency calculation
}).reset_index()

customer_df.columns = [
    'CustomerID',
    'NumInvoices',
    'TotalQuantity',
    'TotalSpending',
    'LastPurchaseDate'
]


#Recency Feature (Days Since Last Purchase)
import datetime as dt

snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

customer_df['Recency'] = (snapshot_date - customer_df['LastPurchaseDate']).dt.days


#Review Engineered Features
print("🔴 Shape of customer-level dataset 🔴", customer_df.shape)
display(customer_df.head())

#5️⃣ Feature Selection
features = [
    'Recency',
    'NumInvoices',
    'TotalQuantity',
    'TotalSpending'
]

X = customer_df[features]
display(X.head())

#Check Feature Correlation
print("\n🔴 Feature Correlation Matrix 🔴")
plt.figure(figsize=(8, 5))
sns.heatmap(X.corr(), annot=True, cmap='coolwarm')
plt.title("Feature Correlation Matrix")
plt.show()


#Feature Scaling (Before Clustering)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Segmentation with the help of K-means Clustering:

In [ ]:
# Component 4
#Now Step 1: Elbow Method (Find Optimal k)

from sklearn.cluster import KMeans
import matplotlib.pyplot as pl

#Computing WCSS
wcss = []

# calculate WCSS for k = 1 to 10
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, random_state=42)
    kmeans.fit(X)        # X = your dataset
    wcss.append(kmeans.inertia_)

# plotting Elbow curve
print('\n🔴 Elbow curve for K-means clustering 🔴')
plt.figure()
plt.plot(range(1, 11), wcss, marker='o')
plt.xlabel('Number of clusters (k)')
plt.ylabel('WCSS')
plt.title('Elbow Method')
plt.show()

#k = 1  to 3 → very sharp drop in WCSS
#k = 3 to 4 → still a noticeable improvement
#After k = 4 → the curve starts flattening (diminishing returns)
# So Elbow point = k = 4


# Run K-means with K=4 as our examined output. So,

kmeans = KMeans(
    n_clusters=4,
    init='k-means++',
    max_iter=300,
    n_init=10,
    random_state=42
)

cluster_labels = kmeans.fit_predict(X_scaled)
customer_df['Cluster'] = cluster_labels
customer_df.head()
customer_df['Cluster'].value_counts()


#Now we will visualize the clusters(scatter plots)
print('\n🔴 Customer Segments 🔴')   
plt.figure()
plt.scatter(
    X_scaled[:, 0],
    X_scaled[:, 1],
    c=cluster_labels
)
plt.xlabel(X.columns[0])
plt.ylabel(X.columns[1])
plt.title('Customer Segments')
plt.show()

#optional plot centriods
print('\n🔴 Clusters with Centroid 🔴')
centroids = kmeans.cluster_centers_

plt.figure()
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=cluster_labels)
plt.scatter(centroids[:, 0], centroids[:, 1], marker='X')
plt.title('Clusters with Centroids')
plt.show()

# we have already confirmed cluster datset and added the 
# cluster labels as a new column to the original dataset before in cell 41 and 44.


#Now, Analyze Each Customer Segment (Statistics)
cluster_mean = customer_df.groupby('Cluster')[features].mean()
cluster_mean
cluster_stats = customer_df.groupby('Cluster')[features].agg(['mean', 'median', 'std'])
cluster_stats


#Visulaizing cluster differences
#1. Bar Chart – Mean Comparison
print('\n🔴 Average Customer Metrics by Cluster 🔴')
cluster_mean.plot(kind='bar')
plt.title('Average Customer Metrics by Cluster')
plt.ylabel('Value')
plt.xticks(rotation=0)
plt.show()

#2.Box Plots – Feature Distribution
print('\n🔴 Feature Distribution by Cluster (Box Plots) 🔴')
for col in features:
    plt.figure()
    sns.boxplot(x='Cluster', y=col, data=customer_df)
    plt.title(f'{col} Distribution by Cluster')
    plt.show()

#3,Pairwise Scatter
print('\n🔴 Customer Segments by Spending & Frequency 🔴')
plt.figure()
plt.scatter(
    customer_df['TotalSpending'],
    customer_df['NumInvoices'],
    c=customer_df['Cluster']
)
plt.xlabel('Total Spending')
plt.ylabel('Number of Invoices')
plt.title('Customer Segments by Spending & Frequency')
plt.show()